# L2b: Errors, Tests, and Debugging a Numerical Program

A program can finish without raising an error and still return an incorrect result. This lab uses a freight-loading calculation with inconsistent mass units to distinguish execution errors, invalid inputs, and incorrect numerical results.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
> * __Check a result against a reference case:__ Compare a function result with a hand-calculated value to detect an incorrect numerical result that does not raise an exception.
> * __Find and correct a unit mismatch:__ Use dimensional analysis to identify the mismatch between metric tonnes and kilograms, then implement the required unit conversion.
> * __Validate and test the function:__ Require both arguments to be finite and strictly positive, and write tests for valid calculations and invalid inputs.

___

## Setup, Data, and Prerequisites

Run the setup cell first. It activates the course environment and loads the code used in this lab.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates `Include.jl` in the notebook's global scope. That file sets local paths, loads the required packages, and includes the lab source code. See the [Julia documentation](https://docs.julialang.org/en/v1/) for details about the language features used here.

Run the setup cell:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

The setup loads [the `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/) and includes [`src/Compute.jl`](src/Compute.jl). You will implement the lab function in that source file.

___

## Task 1: A result that looks plausible

Assume we supervise a loading dock and need to estimate how long a trailer will occupy a particular loading bay. We can estimate the loading time $t$ by dividing the amount of cargo $m$ by the loading rate $r$, or $t = m/r$, where $m$ is in kilograms and $r$ is in kilograms per minute.

> __The reference case:__
>
> The trailer carries 2 metric tons (2 tonnes), which is $2000\;\mathrm{kg}$. At a sustained loading rate of $250\;\mathrm{kg/min}$, the loading time is $2000/250 = 8$ minutes. Let's use this as our benchmark because we know the answer before running the program.

The following function is our first implementation. It runs without raising an error. We will compare its result with the reference case to determine whether the calculation is correct.

In [ ]:
function buggy_loading_time_minutes(cargo_tonnes, loading_rate_kg_min)
    return cargo_tonnes / loading_rate_kg_min
end

buggy_result = buggy_loading_time_minutes(2.0, 250.0)

The function returned `0.008`, which does not agree with the expected value of `8.0`. The function ran successfully, but the reference case shows that the calculation is incorrect.


In [ ]:
expected_minutes = 8.0
diagnostic = (
    result = buggy_result,
    expected = expected_minutes,
    passes = isapprox(buggy_result, expected_minutes),
)

The function divides a mass in metric tonnes by a rate in kilograms per minute. The units in the original expression do not reduce to minutes:

$$
\frac{2\;\mathrm{tonne}}{250\;\mathrm{kg/min}}
= 0.008\;\mathrm{tonne\,min/kg}.
$$

Convert the cargo mass to kilograms before dividing:

$$
\frac{(2\;\mathrm{tonne})(1000\;\mathrm{kg/tonne})}{250\;\mathrm{kg/min}}
= 8\;\mathrm{min}.
$$

The defect is the missing unit conversion. The program does not detect this defect because the numerical division is valid.

___

## Task 2: Implement input validation and unit conversion

The comparison above identifies a missing unit conversion. You will implement the corrected function in [`src/Compute.jl`](src/Compute.jl). Keeping the function in a source file allows the notebook and the validation suite to load the same implementation.

The source file currently contains the function signature, its documentation, and two `TODO` comments.

> __What to write:__
>
> * __Validate both arguments.__ Throw an [`ArgumentError`](https://docs.julialang.org/en/v1/base/base/#Core.ArgumentError) that names the invalid argument when `cargo_tonnes` or `loading_rate_kg_min` is not finite or is not strictly positive. Use [the `isfinite(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.isfinite) to check whether a value is finite.
> * __Convert the units before dividing.__ Convert `cargo_tonnes` to kilograms so that its units match `loading_rate_kg_min`. Divide the converted mass by the loading rate and return the result as a `Float64`.

For this exercise, write the validation directly in the function body. This keeps the input requirements and the calculation visible in one place.

Open the file and complete both `TODO`s. Then restart the kernel and run the notebook from the beginning so that Julia loads the updated source file. The next cell raises a "not implemented yet" error until the function is complete.

In [ ]:
correct_result = cargo_loading_time_minutes(2.0, 250.0)
(correct_result = correct_result, agrees = correct_result == expected_minutes)

___

## Task 3: Test the interface

After the reference case passes, test the complete interface. We will first inspect how the function handles invalid input, then collect the valid and invalid cases in one regression test set.

A zero loading rate would produce `Inf` if the division were allowed to continue. The function should instead throw an error that names the rejected argument.

The following cell catches the error so that we can inspect its message. In an application, catch an error only when the program can respond to it or recover from it:

In [ ]:
caught_message = try
    cargo_loading_time_minutes(2.0, 0.0)
    "no error"
catch error
    sprint(showerror, error)
end

### Regression tests

A regression test checks behavior that must remain correct after the code changes. Julia provides these testing tools through [the `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/), which was loaded during setup.

The test cell uses three macros and one string function:

| Construct | Purpose | Use in this lab |
|:--|:--|:--|
| [The `@testset` macro](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@testset) | Groups related tests and prints a summary. | Groups all checks for the cargo-loading function. |
| [The `@test` macro](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@test) | Passes when an expression evaluates to `true`. | Compares numerical results and checks the error message. |
| [The `@test_throws` macro](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@test_throws) | Passes when an expression raises the specified exception. | Confirms that zero cargo and an infinite loading rate raise [an `ArgumentError`](https://docs.julialang.org/en/v1/base/base/#Core.ArgumentError). |
| [The `occursin(...)` function](https://docs.julialang.org/en/v1/base/strings/#Base.occursin) | Checks whether a string contains specified text. | Confirms that the error message names `loading_rate_kg_min`. |

The first two tests record that the original implementation fails the reference case. The next two tests check the corrected calculation for two valid inputs. The final three tests check the exception type and error message for invalid inputs.

Complete `src/Compute.jl`, restart the kernel, and run the notebook from the beginning. All seven tests should pass.

In [8]:
@testset "errors, tests, and debugging" begin
    @test buggy_result != expected_minutes
    @test !diagnostic.passes
    @test correct_result == expected_minutes
    @test cargo_loading_time_minutes(0.5, 100) == 5.0
    @test_throws ArgumentError cargo_loading_time_minutes(0, 100)
    @test_throws ArgumentError cargo_loading_time_minutes(1, Inf)
    @test occursin("loading_rate_kg_min", caught_message)
end

errors, tests, and debugging: Error During Test at /Users/jeffreyvarner/Desktop/julia_work/CHEME-5800-CourseRepository-Fall-2026/weeks/week-02/L2b/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y146sZmlsZQ==.jl:4
  Test threw exception
  Expression: correct_result == expected_minutes
  UndefVarError: `correct_result` not defined in `Main`
  Suggestion: add an appropriate import or assignment. This global was declared but not assigned.
  Stacktrace:
   [1] top-level scope
     @ ~/Desktop/julia_work/CHEME-5800-CourseRepository-Fall-2026/weeks/week-02/L2b/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y146sZmlsZQ==.jl:2
   [2] macro expansion
     @ ~/.julia/juliaup/julia-1.12.7+0.aarch64.apple.darwin14/Julia-1.12.app/Contents/Resources/julia/share/julia/stdlib/v1.12/Test/src/Test.jl:1777 [inlined]
   [3] macro expansion
     @ ~/Desktop/julia_work/CHEME-5800-CourseRepository-Fall-2026/weeks/week-02/L2b/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y146sZmlsZQ==.jl:4 [inlined]


Test.TestSetException: Some tests did not pass: 2 passed, 3 failed, 2 errored, 0 broken.

___

### Optional: Python implementation

This section is optional. It implements the same unit and input contract in Python so that we can compare how the two languages enforce it.

The Python implementation raises [a `TypeError`](https://docs.python.org/3/library/exceptions.html#TypeError) when an argument has the wrong type and [a `ValueError`](https://docs.python.org/3/library/exceptions.html#ValueError) when a numerical value is invalid. Its helper function performs both checks explicitly.

Julia handles the type check differently. The `::Real` annotation prevents a `String` argument from reaching the function body, so Julia raises [a `MethodError`](https://docs.julialang.org/en/v1/base/base/#Core.MethodError). A numerical value that matches `Real` reaches the validation code and can raise [an `ArgumentError`](https://docs.julialang.org/en/v1/base/base/#Core.ArgumentError).

> __Boolean inputs in Julia:__ [The `Bool` type](https://docs.julialang.org/en/v1/base/numbers/#Core.Bool) is a subtype of `Real`. The call `cargo_loading_time_minutes(true, 100)` therefore passes the type annotation and the current validation checks, treating `true` as the number `1`. Lab `L2d` addresses this issue by rejecting `Bool` values explicitly.

The languages use different syntax and exception types, but they implement the same unit contract and use the same reference case. The next cell displays the commented Python implementation.

In [ ]:
python_source = joinpath(CHEME5800_L2B_ROOT, "src", "cargo_loading.py")
print(read(python_source, String))

The Python tests cover the reference case, integer inputs, invalid numerical values, and invalid types. Run them from the bundle root:

```bash
python -m unittest discover -s weeks/week-02/L2b/src -p 'test_*.py'
```
___

## Summary
In this lab, we used a reference calculation and dimensional analysis to identify a unit error, implemented a loading-time function with input validation and consistent units, and wrote regression tests for valid and invalid inputs.

> __Key Takeaways:__
>
> * **Successful execution does not establish correctness:** A reference calculation is needed to determine whether a numerical result is correct.
> * **Units must be consistent before calculation:** Dimensional analysis identifies the metric-tonnes-to-kilograms conversion required to calculate a loading time in minutes.
> * **Validation and tests define required behavior:** Input checks reject unsupported values, while regression tests verify both valid calculations and expected errors.

We will use reference calculations, unit checks, input validation, and regression tests to verify numerical functions in future problem sets and the projects that follow.
___